# Mintenance AI - YOLO Model Training

Train improved YOLO model on `yolo_dataset_merged_final` dataset

**Time:** 2-4 hours on GPU  
**Dataset:** yolo_dataset_merged_final (4,777 images)  
**Model:** YOLOv11 Medium

## Step 1: Install Dependencies

In [1]:
# Install Ultralytics YOLO
!pip install ultralytics pyyaml --quiet

# Verify installation
from ultralytics import YOLO
print("✅ Ultralytics installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Ultralytics installed successfully


## Step 2: Enable GPU

**Important:** Go to Runtime → Change runtime type → Select "T4 GPU" → Save

In [2]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ No GPU - training will be slow on CPU")

CUDA available: True
GPU: Tesla T4
GPU Memory: 14.74 GB


## Step 3: Mount Google Drive

**Action Required:** Click the link, authorize, and paste the code when prompted

In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

Mounted at /content/drive
✅ Google Drive mounted


## Step 4: Extract Dataset

**Make sure:** `yolo_dataset_merged_final.zip` is in `/content/drive/MyDrive/YOLO_Training/`

In [4]:
import zipfile
import os

# Paths
zip_path = "/content/drive/MyDrive/YOLO_Training/yolo_dataset_merged_final.zip"
extract_path = "/content/dataset"

# Create extraction directory
os.makedirs(extract_path, exist_ok=True)

# Extract dataset
print("📦 Extracting dataset... (2-5 minutes)")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extracted!")

📦 Extracting dataset... (2-5 minutes)
✅ Dataset extracted!


## Step 4.5: Fix data.yaml Paths

**Important:** Fix Windows paths in data.yaml to work with Colab

In [5]:
# Fix data.yaml paths for Colab
import yaml
import os

dataset_path = "/content/dataset/yolo_dataset_merged_final"
data_yaml_path = f"{dataset_path}/data.yaml"

print("🔧 Fixing data.yaml paths for Colab...")

# First, check if dataset was extracted
if not os.path.exists(dataset_path):
    print(f"❌ Dataset not found at: {dataset_path}")
    print("\n🔍 Checking what exists in /content/dataset...")

    extract_base = "/content/dataset"
    if os.path.exists(extract_base):
        items = os.listdir(extract_base)
        print(f"   Found {len(items)} items in /content/dataset:")
        for item in items[:10]:  # Show first 10
            item_path = os.path.join(extract_base, item)
            if os.path.isdir(item_path):
                print(f"   📁 {item}/")
            else:
                print(f"   📄 {item}")
        if len(items) > 10:
            print(f"   ... and {len(items) - 10} more items")

        # Try to find the dataset folder
        possible_paths = [
            "/content/dataset/yolo_dataset_merged_final",
            "/content/dataset/dataset",
            "/content/dataset/train",  # Maybe extracted directly
        ]

        for possible_path in possible_paths:
            if os.path.exists(possible_path):
                print(f"\n💡 Found dataset at: {possible_path}")
                dataset_path = possible_path
                data_yaml_path = f"{dataset_path}/data.yaml"
                break
        else:
            # Check if train/images exists directly
            if os.path.exists("/content/dataset/train/images"):
                print("\n💡 Found train/images directly in /content/dataset")
                # Create dataset structure
                dataset_path = "/content/dataset"
                data_yaml_path = f"{dataset_path}/data.yaml"
            else:
                print("\n❌ Could not find dataset structure!")
                print("   Please re-run Step 4 (Extract Dataset)")
                raise FileNotFoundError(f"Dataset not extracted: {dataset_path}")
    else:
        print("   /content/dataset doesn't exist!")
        print("   Please run Step 4 (Extract Dataset) first!")
        raise FileNotFoundError(f"Dataset directory not found: {extract_base}")

# Check if data.yaml exists, if not create it
if os.path.exists(data_yaml_path):
    print("📋 Found existing data.yaml, updating paths...")
    # Read current data.yaml
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    # Update paths to use absolute Colab paths
    data_config['train'] = f"{dataset_path}/train/images"
    data_config['val'] = f"{dataset_path}/val/images"
    if 'test' in data_config:
        data_config['test'] = f"{dataset_path}/test/images"

    # Set path to dataset root
    if 'path' in data_config:
        data_config['path'] = dataset_path
else:
    print("📋 data.yaml not found, creating new one...")

    # Count images to verify dataset structure
    train_path = f"{dataset_path}/train/images"
    val_path = f"{dataset_path}/val/images"
    test_path = f"{dataset_path}/test/images"

    train_count = len([f for f in os.listdir(train_path) if f.endswith('.jpg')]) if os.path.exists(train_path) else 0
    val_count = len([f for f in os.listdir(val_path) if f.endswith('.jpg')]) if os.path.exists(val_path) else 0
    test_count = len([f for f in os.listdir(test_path) if f.endswith('.jpg')]) if os.path.exists(test_path) else 0

    # Create new data.yaml with standard building defect classes
    data_config = {
        'path': dataset_path,
        'train': f"{dataset_path}/train/images",
        'val': f"{dataset_path}/val/images",
        'test': f"{dataset_path}/test/images",
        'nc': 15,  # Number of classes
        'names': [
            'general_damage',
            'cracks',
            'mold',
            'water_damage',
            'structural_damage',
            'electrical_issues',
            'plumbing_issues',
            'roofing_damage',
            'window_damage',
            'door_damage',
            'floor_damage',
            'wall_damage',
            'ceiling_damage',
            'hvac_issues',
            'insulation_issues'
        ]
    }

    print(f"   Created data.yaml with {data_config['nc']} classes")

# Save fixed/created data.yaml
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("✅ data.yaml ready!")
print(f"   Train: {data_config['train']}")
print(f"   Val: {data_config['val']}")
if 'test' in data_config:
    print(f"   Test: {data_config['test']}")

# Verify paths exist and count images
print("\n🔍 Verifying paths...")
for key in ['train', 'val']:
    path = data_config[key]
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.endswith('.jpg')])
        print(f"   ✅ {key}: {count} images found")
    else:
        print(f"   ❌ {key}: Path not found - {path}")
        print(f"      Make sure dataset was extracted correctly in Step 4!")

# Also check test if it exists
if 'test' in data_config and os.path.exists(data_config['test']):
    test_count = len([f for f in os.listdir(data_config['test']) if f.endswith('.jpg')])
    print(f"   ✅ test: {test_count} images found")

🔧 Fixing data.yaml paths for Colab...
📋 Found existing data.yaml, updating paths...
✅ data.yaml ready!
   Train: /content/dataset/yolo_dataset_merged_final/train/images
   Val: /content/dataset/yolo_dataset_merged_final/val/images
   Test: /content/dataset/yolo_dataset_merged_final/test/images

🔍 Verifying paths...
   ✅ train: 4692 images found
   ✅ val: 1785 images found
   ✅ test: 398 images found


## Step 4.6: Fix Invalid Class IDs (CRITICAL!)

**IMPORTANT:** Your dataset has labels with invalid class IDs (58, 66, 43, 70, etc.) that exceed the valid range (0-14). This causes images to be skipped during training, leading to poor performance (18.87% mAP50).

**This step fixes the dataset by removing invalid class IDs from label files.**

In [ ]:
# Fix invalid class IDs in dataset
# This removes labels with class IDs > 14 (invalid for 15-class dataset)
import os
from pathlib import Path
import yaml

DATASET_DIR = Path("/content/dataset/yolo_dataset_merged_final")
MAX_CLASS_ID = 14  # For 15 classes (0-14)

print("="*70)
print(" FIXING DATASET CLASS ID ISSUES")
print("="*70)
print()

# Load data.yaml to verify configuration
data_yaml_path = DATASET_DIR / "data.yaml"
if not data_yaml_path.exists():
    print(f"❌ Error: data.yaml not found at {data_yaml_path}")
    print("   Run Step 4.5 first!")
else:
    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    print(f"📋 Dataset configuration:")
    print(f"   Classes: {data['nc']}")
    print(f"   Valid class IDs: 0 to {data['nc']-1}")
    print()

# Statistics
stats = {
    'train': {'total': 0, 'fixed': 0, 'removed': 0, 'invalid_ids': set()},
    'val': {'total': 0, 'fixed': 0, 'removed': 0, 'invalid_ids': set()},
    'test': {'total': 0, 'fixed': 0, 'removed': 0, 'invalid_ids': set()}
}

# Process each split
for split in ['train', 'val', 'test']:
    labels_dir = DATASET_DIR / split / 'labels'
    
    if not labels_dir.exists():
        print(f"⚠️  {split}/labels directory not found, skipping...")
        continue
    
    print(f"🔍 Processing {split} labels...")
    label_files = list(labels_dir.glob('*.txt'))
    stats[split]['total'] = len(label_files)
    
    for label_file in label_files:
        try:
            with open(label_file, 'r') as f:
                lines = f.readlines()
            
            valid_lines = []
            file_has_invalid = False
            
            for line in lines:
                line = line.strip()
                if not line:
                    continue
                
                parts = line.split()
                if len(parts) < 5:
                    continue
                
                class_id = int(parts[0])
                
                # Check if class ID is valid
                if class_id < 0 or class_id > MAX_CLASS_ID:
                    stats[split]['invalid_ids'].add(class_id)
                    file_has_invalid = True
                    stats[split]['removed'] += 1
                    continue  # Skip invalid class IDs
                
                # Keep valid lines
                valid_lines.append(line)
            
            # Write back if file was modified
            if file_has_invalid:
                with open(label_file, 'w') as f:
                    f.write('\n'.join(valid_lines) + '\n')
                stats[split]['fixed'] += 1
        
        except Exception as e:
            print(f"   ⚠️  Error processing {label_file.name}: {e}")

# Summary
print()
print("="*70)
print(" SUMMARY")
print("="*70)
total_fixed = sum(s['fixed'] for s in stats.values())
total_removed = sum(s['removed'] for s in stats.values())
all_invalid_ids = set()
for s in stats.values():
    all_invalid_ids.update(s['invalid_ids'])

print(f"✅ Fixed {total_fixed} label files")
print(f"✅ Removed {total_removed} invalid annotations")
if all_invalid_ids:
    print(f"⚠️  Invalid class IDs found: {sorted(all_invalid_ids)}")
    print(f"   These were from the original dataset (81 classes)")
    print(f"   and should have been remapped to 0-14")
print()

if total_fixed > 0:
    print("✅ Dataset fixed! Ready for training.")
    print("   These invalid labels were causing poor performance (18.87% mAP50)")
    print("   Now re-run training - should get much better results!")
else:
    print("ℹ️  No issues found. Dataset is ready for training.")

## Step 5: Verify Dataset

In [6]:
# Check dataset structure
dataset_path = "/content/dataset/yolo_dataset_merged_final"

train_images = len([f for f in os.listdir(f"{dataset_path}/train/images") if f.endswith('.jpg')])
val_images = len([f for f in os.listdir(f"{dataset_path}/val/images") if f.endswith('.jpg')])
test_images = len([f for f in os.listdir(f"{dataset_path}/test/images") if f.endswith('.jpg')])

print("📊 Dataset Statistics:")
print(f"   Train: {train_images} images")
print(f"   Val: {val_images} images")
print(f"   Test: {test_images} images")
print(f"   Total: {train_images + val_images + test_images} images")

# Check data.yaml
data_yaml = f"{dataset_path}/data.yaml"
if os.path.exists(data_yaml):
    print(f"\n✅ data.yaml found")
    import yaml
    with open(data_yaml, 'r') as f:
        config = yaml.safe_load(f)
        print(f"   Classes: {config['nc']}")
else:
    print(f"\n❌ data.yaml not found!")

📊 Dataset Statistics:
   Train: 4692 images
   Val: 1785 images
   Test: 398 images
   Total: 6875 images

✅ data.yaml found
   Classes: 15


## Step 6: Configure Training

In [ ]:
# Training configuration - IMPROVED VERSION
# Previous training got 20.7% mAP50 (worse than current 27.1%)
# This improved config should perform better

# Choose configuration:
# 'large' = yolo11l.pt, 150 epochs (RECOMMENDED - best balance)
# 'tuned' = yolo11m.pt, 150 epochs (faster, still improved)
# 'xlarge' = yolo11x.pt, 200 epochs (best performance, slowest)
CONFIG_CHOICE = 'large'  # Change this to try different configs

if CONFIG_CHOICE == 'large':
    TRAINING_CONFIG = {
        'data_yaml': '/content/dataset/yolo_dataset_merged_final/data.yaml',
        'model': 'yolo11l.pt',  # Large model - better accuracy
        'epochs': 150,  # More epochs
        'imgsz': 640,
        'batch': 12,  # Reduced for larger model
        'name': 'mintenance_ai_v4_large',
        'project': 'runs/train',
        'patience': 30,  # More patience
        'lr0': 0.005,  # Lower learning rate
    }
elif CONFIG_CHOICE == 'tuned':
    TRAINING_CONFIG = {
        'data_yaml': '/content/dataset/yolo_dataset_merged_final/data.yaml',
        'model': 'yolo11m.pt',  # Medium model
        'epochs': 150,
        'imgsz': 640,
        'batch': 16,
        'name': 'mintenance_ai_v4_tuned',
        'project': 'runs/train',
        'patience': 30,
        'lr0': 0.005,
    }
else:  # xlarge
    TRAINING_CONFIG = {
        'data_yaml': '/content/dataset/yolo_dataset_merged_final/data.yaml',
        'model': 'yolo11x.pt',  # Extra Large - best accuracy
        'epochs': 200,
        'imgsz': 640,
        'batch': 8,  # Smaller batch for large model
        'name': 'mintenance_ai_v4_xlarge',
        'project': 'runs/train',
        'patience': 40,
        'lr0': 0.003,  # Very low learning rate
    }

print("⚙️ IMPROVED Training Configuration:")
print(f"   Config: {CONFIG_CHOICE}")
for key, value in TRAINING_CONFIG.items():
    print(f"   {key}: {value}")

print("\n💡 Improvements:")
print("   - Larger model (better accuracy)")
print("   - More epochs (150-200 vs 100)")
print("   - Lower learning rate (better convergence)")
print("   - More patience (don't stop early)")
print("   - Enhanced augmentation (better generalization)")

print("\n📊 Expected Results:")
if CONFIG_CHOICE == 'large':
    print("   Expected mAP50: 30-40% (vs current 27.1%)")
    print("   Training time: 4-5 hours")
elif CONFIG_CHOICE == 'tuned':
    print("   Expected mAP50: 25-35% (vs current 27.1%)")
    print("   Training time: 4 hours")
else:
    print("   Expected mAP50: 35-45% (vs current 27.1%)")
    print("   Training time: 6-8 hours")

⚙️ Training Configuration:
   data_yaml: /content/dataset/yolo_dataset_merged_final/data.yaml
   model: yolo11m.pt
   epochs: 100
   imgsz: 640
   batch: 16
   name: mintenance_ai_v3
   project: runs/train

💡 If model download fails, try:
   - 'yolov8m.pt' (YOLOv8 Medium)
   - 'yolov10m.pt' (YOLOv10 Medium)
   - 'yolo11m.pt' (YOLOv11 Medium)


## Step 7: Start Training

**⏱️ This will take 2-4 hours. Keep this tab open!**

In [ ]:
# Load model and train with IMPROVED configuration
from ultralytics import YOLO
import os

model_name = TRAINING_CONFIG['model']
print(f"🔄 Loading model: {model_name}")

# Check if model exists, if not it will be downloaded automatically
if not os.path.exists(model_name):
    print(f"   Model not found locally, downloading...")
    print(f"   This may take 1-2 minutes on first run")

try:
    # YOLO will automatically download the model if it doesn't exist
    model = YOLO(model_name)
    print(f"✅ Model loaded: {model_name}")

    # Verify model file exists now
    if os.path.exists(model_name):
        size_mb = os.path.getsize(model_name) / (1024 * 1024)
        print(f"   Model size: {size_mb:.2f} MB")
    else:
        # Sometimes model is cached elsewhere, check Ultralytics cache
        from pathlib import Path
        cache_dir = Path.home() / '.ultralytics' / 'weights'
        if cache_dir.exists():
            cached_models = list(cache_dir.glob('*.pt'))
            if cached_models:
                print(f"   Model cached at: {cached_models[0]}")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\n💡 Trying alternative: Download model explicitly")
    try:
        from ultralytics.utils.downloads import download
        model_path = download(model_name)
        model = YOLO(model_path)
        print(f"✅ Model downloaded and loaded: {model_path}")
    except Exception as e2:
        print(f"❌ Failed to download model: {e2}")
        raise

# Start IMPROVED training
print("\n🚀 Starting IMPROVED training...")
print(f"   Model: {TRAINING_CONFIG['model']}")
print(f"   Epochs: {TRAINING_CONFIG['epochs']}")
print(f"   Batch: {TRAINING_CONFIG['batch']}")
print(f"   Patience: {TRAINING_CONFIG['patience']}")
print(f"   Learning Rate: {TRAINING_CONFIG['lr0']}")
print(f"   This will take 4-6 hours on GPU")
print(f"   Keep this tab open!")
print()

results = model.train(
    data=TRAINING_CONFIG['data_yaml'],
    epochs=TRAINING_CONFIG['epochs'],
    imgsz=TRAINING_CONFIG['imgsz'],
    batch=TRAINING_CONFIG['batch'],
    device=0,  # GPU
    project=TRAINING_CONFIG['project'],
    name=TRAINING_CONFIG['name'],
    patience=TRAINING_CONFIG['patience'],  # More patience
    save=True,
    plots=True,
    verbose=True,
    # Optimized learning rate
    lr0=TRAINING_CONFIG['lr0'],  # Lower learning rate
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    optimizer='AdamW',
    # Enhanced augmentation (more variation)
    hsv_h=0.02,  # More color variation
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15,  # More rotation
    translate=0.15,  # More translation
    scale=0.6,  # More scaling
    mosaic=1.0,  # Enable mosaic augmentation
    mixup=0.1,  # Add mixup augmentation
    copy_paste=0.1,  # Copy-paste augmentation
    # Warmup settings
    warmup_epochs=5,  # More warmup
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    # Loss weights
    cls=0.5,
    box=7.5,
)

print("\n✅ Training complete!")
print(f"\n📊 Next: Run Step 8 to validate and compare with current model (27.1% mAP50)")

🔄 Loading model: yolo11m.pt
   Model not found locally, downloading...
   This may take 1-2 minutes on first run
✅ Model loaded: yolo11m.pt
   Model size: 38.80 MB

🚀 Starting training...
   This will take 2-4 hours on GPU
   Keep this tab open!

Ultralytics 8.4.6 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/yolo_dataset_merged_final/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, l

## Step 8: Validate Model

In [ ]:
# Validate trained model
from ultralytics import YOLO
import os

# Define paths - use the name from training config
# This will match the name set in Step 6 (Configuration)
if 'TRAINING_CONFIG' in globals():
    name = TRAINING_CONFIG['name']
else:
    # Fallback: try to detect from config choice
    name = 'mintenance_ai_v4_large'  # Default to large config

data_yaml = '/content/dataset/yolo_dataset_merged_final/data.yaml'

# Check multiple possible locations (Ultralytics may save to different paths)
possible_paths = [
    '/content/runs/detect/runs/train/mintenance_ai_v3/weights/best.pt',  # Actual location from training
    'runs/train/mintenance_ai_v3/weights/best.pt',
    '/content/runs/train/mintenance_ai_v3/weights/best.pt',
]

best_model_path = None
for path in possible_paths:
    if os.path.exists(path):
        best_model_path = path
        print(f"✅ Found model at: {path}")
        break

if not best_model_path:
    print("🔍 Searching for model files...")
    # Search for best.pt files
    search_dirs = [
        '/content/runs/detect/runs/train',
        '/content/runs/train',
        'runs/train',
    ]
    
    for search_dir in search_dirs:
        if os.path.exists(search_dir):
            for root, dirs, files in os.walk(search_dir):
                if 'best.pt' in files:
                    best_model_path = os.path.join(root, 'best.pt')
                    print(f"✅ Found model at: {best_model_path}")
                    break
            if best_model_path:
                break

if best_model_path and os.path.exists(best_model_path):
    trained_model = YOLO(best_model_path)

    print("\n🔍 Validating model on validation set...")
    metrics = trained_model.val(
        data=data_yaml,
        split='val'
    )

    print("\n📊 Validation Results:")
    print(f"   mAP50: {metrics.box.map50:.2%}")
    print(f"   mAP50-95: {metrics.box.map:.2%}")
    print(f"   Precision: {metrics.box.mp:.2%}")
    print(f"   Recall: {metrics.box.mr:.2%}")

    # Compare with target
    print("\n🎯 Target vs Actual:")
    print(f"   mAP50 Target: 45-55%")
    print(f"   mAP50 Actual: {metrics.box.map50:.2%}")
    if metrics.box.map50 >= 0.35:
        print("   ✅ Improvement achieved!")
    else:
        print("   ⚠️  May need more training or data")
else:
    print("❌ Model not found in any expected location")
    print("\n💡 Check training output for actual save path")

🔍 Looking for model at: runs/train/mintenance_ai_v3/weights/best.pt
❌ Model not found
   Expected path: runs/train/mintenance_ai_v3/weights/best.pt

💡 Check training completed:
   Training directory not found: runs/train


## Step 9: Save to Google Drive

**Download from Google Drive to your local machine after this step**

In [ ]:
# Save model to Google Drive
import shutil
import os

# Define paths - use the name from training config
if 'TRAINING_CONFIG' in globals():
    name = TRAINING_CONFIG['name']
    model_filename = f"{name}_model.pt"
else:
    # Fallback: try to detect from config choice
    name = 'mintenance_ai_v4_large'
    model_filename = f"{name}_model.pt"

drive_backup_path = f"/content/drive/MyDrive/YOLO_Training/{model_filename}"

# Check multiple possible locations (Ultralytics may save to different paths)
possible_paths = [
    '/content/runs/detect/runs/train/mintenance_ai_v3/weights/best.pt',  # Actual location from training
    'runs/train/mintenance_ai_v3/weights/best.pt',
    '/content/runs/train/mintenance_ai_v3/weights/best.pt',
]

best_model_path = None
for path in possible_paths:
    if os.path.exists(path):
        best_model_path = path
        print(f"✅ Found model at: {path}")
        break

if not best_model_path:
    print("🔍 Searching for model files...")
    # Search for best.pt files
    search_dirs = [
        '/content/runs/detect/runs/train',
        '/content/runs/train',
        'runs/train',
    ]
    
    for search_dir in search_dirs:
        if os.path.exists(search_dir):
            for root, dirs, files in os.walk(search_dir):
                if 'best.pt' in files:
                    best_model_path = os.path.join(root, 'best.pt')
                    print(f"✅ Found model at: {best_model_path}")
                    break
            if best_model_path:
                break

if best_model_path and os.path.exists(best_model_path):
    shutil.copy(best_model_path, drive_backup_path)
    size_mb = os.path.getsize(best_model_path) / (1024 * 1024)
    print(f"\n✅ Model saved to Google Drive!")
    print(f"   Size: {size_mb:.2f} MB")
    print(f"   Path: {drive_backup_path}")
    print("\n📥 Next: Download from Google Drive to your local machine")
    print("   Location: MyDrive/YOLO_Training/mintenance_ai_v3_model.pt")
else:
    print("❌ Model not found in any expected location")
    print("\n💡 Check training output - model should be at:")
    print("   /content/runs/detect/runs/train/mintenance_ai_v3/weights/best.pt")

🔍 Looking for model at: runs/train/mintenance_ai_v3/weights/best.pt
❌ Model not found
   Expected path: runs/train/mintenance_ai_v3/weights/best.pt


## Step 10: Download Results (Optional)

Download training plots, metrics, and visualizations

In [ ]:
# Download training results
from google.colab import files
import zipfile
from pathlib import Path
import os

# Define paths - use the name from training config
if 'TRAINING_CONFIG' in globals():
    name = TRAINING_CONFIG['name']
else:
    # Fallback: try to detect from config choice
    name = 'mintenance_ai_v4_large'

zip_path = f"/content/{name}_results.zip"

# Check multiple possible locations
possible_dirs = [
    Path('/content/runs/detect/runs/train/mintenance_ai_v3'),  # Actual location from training
    Path('runs/train/mintenance_ai_v3'),
    Path('/content/runs/train/mintenance_ai_v3'),
]

results_dir = None
for dir_path in possible_dirs:
    if dir_path.exists():
        results_dir = dir_path
        print(f"✅ Found results at: {results_dir}")
        break

if not results_dir:
    print("🔍 Searching for training results...")
    # Search for training directories
    search_dirs = [
        '/content/runs/detect/runs/train',
        '/content/runs/train',
        'runs/train',
    ]
    
    for search_dir in search_dirs:
        if os.path.exists(search_dir):
            for item in os.listdir(search_dir):
                item_path = Path(search_dir) / item
                if item_path.is_dir() and (item_path / 'weights').exists():
                    results_dir = item_path
                    print(f"✅ Found results at: {results_dir}")
                    break
            if results_dir:
                break

if results_dir and results_dir.exists():
    print("📦 Creating ZIP file...")
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for file in results_dir.rglob('*'):
            if file.is_file():
                zipf.write(file, file.relative_to(results_dir.parent))

    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"✅ ZIP created ({size_mb:.2f} MB)")
    files.download(zip_path)
    print("✅ Training results downloaded!")
else:
    print("❌ Results not found")
    print("\n💡 Check training output - results should be at:")
    print("   /content/runs/detect/runs/train/mintenance_ai_v3")

🔍 Looking for results at: runs/train/mintenance_ai_v3
❌ Results not found
   Expected path: runs/train/mintenance_ai_v3
